## Frozen-feature screening

Screens pretrained backbones and input variants with logistic-regression heads on the identity-cluster folds of `manifest.csv` (from `tongue-dataprep`, cv mode). Originals only; augmentations are not used.

- **Embeddings**: one per (backbone, input variant, size), mean over 10 views (5 crops at 90% x horizontal flip); cached in `features/`.
- **Head**: StandardScaler + LogisticRegression; `C` chosen by 3-fold group CV over training clusters; threshold by the same rule as the training notebook (recall s.t. precision >= 0.90, ties to the middle).
- **Order**: (1) backbones on `circle`; (2) inputs `full`, `circle`, `seg`, `segcrop` on the best two; (3) gray-world white balance; (4) 224 vs 448; (5) ensemble of the two finalists. Every finalist is also scored on `corners` (lips, clothing, background only).
- **Honest estimate (nested selection)**: the leaderboard picks its winner on all 106 people, so re-scoring it on other folds would reuse them. Instead, for each fresh outer fold (`fold_s2`, StratifiedGroupKFold seed 2024) the whole selection is re-run on the outer training people only (scored by inner group CV) and the chosen pipeline predicts the held-out people. Its pooled AUC, compared with the fixed frozen ResNet-50 circle pipeline on the same folds, is the number to trust.
- **Outputs**: `masks/<family>.png`, `screen_oof.csv`, `screen_summary.csv`, `screen_choice.json`, and for the final pipeline `final_oof.csv`, `final_summary.csv`, `final_pipeline.json`. Masks are seeded, so they are identical on every run; features that use them are cached under the masks' digest.
- `SMOKE=1` (local CPU check): 100 originals, ResNet-50 and DINOv2-S without pretrained weights, 2 views, size 224 only.

In [7]:
import os, re, json, random, gc, hashlib
import psutil
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import torch
import timm
import torchvision
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, precision_score, recall_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

MANIFEST = Path(os.environ.get('MANIFEST', '/kaggle/input/notebooks/edamhakim/tongue-dataprep/manifest.csv'))
OUT = Path(os.environ.get('OUTPUT_BASE', '/kaggle/working'))
SMOKE = os.environ.get('SMOKE') == '1'
SEED, N_FOLDS = 42, 5
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
(OUT / 'features').mkdir(parents=True, exist_ok=True)
(OUT / 'masks').mkdir(parents=True, exist_ok=True)
print('Device:', device, '| torch', torch.__version__, '| timm', timm.__version__, '| smoke', SMOKE)

Device: cuda | torch 2.11.0+cu130 | timm 1.0.30 | smoke False


In [8]:
# Tongue masks and white balance (ported verbatim from analysis/tongue_masks.py, eye-checked on 40 originals)
MIN_AREA, MAX_AREA, WORK = 0.20, 0.85, 300  # the 3% sure-background border caps masks at ~0.88


def circle_mask(h, w):
    yy, xx = np.mgrid[-1:1:h * 1j, -1:1:w * 1j]
    return ((xx ** 2 + yy ** 2) <= 0.85 ** 2).astype(np.uint8)


def gray_world(img_bgr):
    img = img_bgr.astype(np.float32)
    means = img.reshape(-1, 3).mean(0)
    return np.clip(img * (means.mean() / np.maximum(means, 1e-6)), 0, 255).round().astype(np.uint8)


def tongue_mask(img_bgr):
    """(uint8 0/1 mask at image size, fell_back)."""
    h, w = img_bgr.shape[:2]
    small = cv2.resize(img_bgr, (WORK, WORK))
    yy, xx = np.mgrid[-1:1:WORK * 1j, -1:1:WORK * 1j]
    gc = np.full((WORK, WORK), cv2.GC_PR_BGD, np.uint8)
    # Crops are tight and centred: position seeds work for coated, pale and purple tongues alike
    gc[(xx / 0.8) ** 2 + (yy / 0.9) ** 2 <= 1] = cv2.GC_PR_FGD
    gc[(xx / 0.4) ** 2 + (yy / 0.45) ** 2 <= 1] = cv2.GC_FGD
    b = int(0.03 * WORK)
    gc[:b], gc[-b:], gc[:, :b], gc[:, -b:] = cv2.GC_BGD, cv2.GC_BGD, cv2.GC_BGD, cv2.GC_BGD
    fg = np.zeros((WORK, WORK), np.uint8)
    if (gc == cv2.GC_PR_FGD).any():
        try:
            cv2.setRNGSeed(0)  # GrabCut's k-means init uses OpenCV's RNG: same masks on every run and image order
            cv2.grabCut(small, gc, None, np.zeros((1, 65)), np.zeros((1, 65)), 5, cv2.GC_INIT_WITH_MASK)
            fg = ((gc == cv2.GC_FGD) | (gc == cv2.GC_PR_FGD)).astype(np.uint8)
        except cv2.error:
            pass  # degenerate colour models: treated like an empty mask, i.e. fallback
    n, lab, stats, _ = cv2.connectedComponentsWithStats(fg)
    if n > 1:
        fg = (lab == 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])).astype(np.uint8)
        holes = fg.copy()
        cv2.floodFill(holes, np.zeros((WORK + 2, WORK + 2), np.uint8), (0, 0), 1)
        fg = fg | (1 - holes)
    if not MIN_AREA <= fg.mean() <= MAX_AREA:
        return circle_mask(h, w), True
    return cv2.resize(fg, (w, h), interpolation=cv2.INTER_NEAREST), False


def fresh_folds(orig):
    """Fold 0-4 per original from StratifiedGroupKFold(5, shuffle, 2024) on label grouped by group_cluster."""
    folds = pd.Series(-1, index=orig.index)
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=2024)
    for k, (_, te) in enumerate(sgkf.split(orig, orig['label'], orig['group_cluster'])):
        folds.iloc[te] = k
    return folds

In [9]:
root = MANIFEST.parent
orig = pd.read_csv(MANIFEST).query('is_orig == 1').reset_index(drop=True)
orig['fold_s2'] = fresh_folds(orig)
assert orig.groupby('group_cluster').fold_s2.nunique().max() == 1
if SMOKE:  # 10 per class per cluster fold, so every fold keeps both classes
    orig = orig.groupby(['fold_cluster', 'label']).head(10).reset_index(drop=True)
print(f'{len(orig)} originals, {orig.group_cluster.nunique()} identity clusters')

fell_back = 0
for fam, f in zip(orig.family, orig.file):
    # always regenerated (seeded, ~1 min): masks left in /kaggle/working by an older run must not be reused
    m, fb = tongue_mask(cv2.imread(str(root / f)))
    cv2.imwrite(str(OUT / 'masks' / f'{fam}.png'), m * 255)
    fell_back += fb
print('masks written; fallback to circle:', fell_back)
MASK_TAG = hashlib.md5(b''.join((OUT / 'masks' / f'{fam}.png').read_bytes() for fam in orig.family)).hexdigest()[:8]
print('mask digest:', MASK_TAG)

497 originals, 106 identity clusters
masks written; fallback to circle: 0
mask digest: 099d67c9


In [10]:
MEAN_RGB = np.array([124, 116, 104], np.uint8)  # ImageNet mean colour = 0 after normalization


def prepare(file, family, variant):
    """RGB PIL image for one input variant: full, circle, seg, segcrop, bg, corners; '+gw' = gray-world first."""
    base, gw = variant.split('+')[0], variant.endswith('+gw')
    bgr = cv2.imread(str(root / file))
    if gw:
        bgr = gray_world(bgr)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    h, w = rgb.shape[:2]
    if base == 'full':
        keep = np.ones((h, w), np.uint8)
    elif base == 'circle':
        keep = circle_mask(h, w)
    elif base in ('seg', 'segcrop', 'bg'):
        mask = cv2.imread(str(OUT / 'masks' / f'{family}.png'), 0)
        if mask is None or mask.shape != (h, w):
            raise FileNotFoundError(f'tongue mask missing or wrong size for family {family}')
        keep = (mask > 0).astype(np.uint8)
        if base == 'bg':  # surroundings: outside the tongue mask dilated by ~5% of the crop (lips, skin, teeth)
            k = max(3, int(0.05 * h)) | 1
            keep = 1 - cv2.dilate(keep, np.ones((k, k), np.uint8))
    elif base == 'corners':  # outside the inscribed circle, as in the training notebook's periphery probe
        yy, xx = np.mgrid[-1:1:h * 1j, -1:1:w * 1j]
        keep = ((xx ** 2 + yy ** 2) >= 1.0).astype(np.uint8)
    else:
        raise ValueError(variant)
    rgb = np.where(keep[..., None] == 1, rgb, MEAN_RGB)
    if base == 'segcrop':
        ys, xs = np.where(keep)
        my, mx = int(0.05 * h), int(0.05 * w)
        rgb = rgb[max(0, ys.min() - my):ys.max() + my + 1, max(0, xs.min() - mx):xs.max() + mx + 1]
    return Image.fromarray(rgb.astype(np.uint8))

In [11]:
BACKBONES = {'dinov2_s': 'vit_small_patch14_dinov2.lvd142m', 'dinov2_b': 'vit_base_patch14_dinov2.lvd142m',
             'convnext_s': 'convnext_small.fb_in22k_ft_in1k', 'effv2_s': 'tf_efficientnetv2_s.in21k_ft_in1k',
             'resnet50': 'torchvision'}
N_VIEWS = 2 if SMOKE else 10
_models = {}


def load_backbone(name):
    if name not in _models:
        if name == 'resnet50':
            m = resnet50(weights=None if SMOKE else ResNet50_Weights.IMAGENET1K_V2)
            m.fc = torch.nn.Identity()
            mean, std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
        else:
            kw = {'dynamic_img_size': True} if name.startswith('dinov2') else {}
            m = timm.create_model(BACKBONES[name], pretrained=not SMOKE, num_classes=0, **kw)
            cfg = timm.data.resolve_model_data_config(m)
            mean, std = cfg['mean'], cfg['std']
        _models.clear()  # keep one backbone in memory; free the old one before loading the next
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        _models[name] = (m.eval().to(device), transforms.Normalize(mean, std))
    return _models[name]


def views(img, size):
    """N_VIEWS tensors: 5 crops at 90% (centre + corners) x horizontal flip; smoke = centre x flip."""
    big = transforms.Resize((round(size / 0.9), round(size / 0.9)), interpolation=transforms.InterpolationMode.BICUBIC)(img)
    crops = transforms.FiveCrop(size)(big)
    crops = [crops[4]] if SMOKE else list(crops)
    out = []
    for c in crops:
        t = transforms.ToTensor()(c)
        out += [t, torch.flip(t, dims=[2])]
    return out


def hsv_hist(img):
    """960-bin (30 hue x 32 saturation) L1-normalized histogram, ignoring pixels filled with MEAN_RGB."""
    rgb = np.array(img)
    keep = (~(rgb == MEAN_RGB).all(-1)).astype(np.uint8)
    h = cv2.calcHist([cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)], [0, 1], keep, [30, 32], [0, 180, 0, 256]).ravel()
    return h / max(h.sum(), 1)


@torch.no_grad()
def extract(backbone, variant, size):
    """(n_originals, n_features) mean embedding over views; cached per (backbone, variant, size)."""
    tag = f'__m{MASK_TAG}' if variant.split('+')[0] in ('seg', 'segcrop', 'bg') else ''
    path = OUT / 'features' / f'{backbone}__{variant}__{size}{tag}{"_smoke" if SMOKE else ""}.npy'
    files = path.with_suffix('.files.txt')
    if path.exists():
        assert files.exists() and files.read_text().split('\n') == list(orig.file), f'stale feature cache {path}: rows differ from this manifest'
        return np.load(path)
    if backbone == 'hsv':  # low-level baseline: hue x saturation histogram of the kept (non-fill) pixels
        X = np.stack([hsv_hist(prepare(file, fam, variant)) for file, fam in zip(orig.file, orig.family)])
    else:
        model, norm = load_backbone(backbone)
        feats = []
        for file, fam in zip(orig.file, orig.family):
            x = torch.stack([norm(v) for v in views(prepare(file, fam, variant), size)]).to(device)
            feats.append(model(x).float().mean(0).cpu().numpy())
        X = np.stack(feats)
    np.save(path, X)
    files.write_text('\n'.join(orig.file))
    gpu = f', GPU {torch.cuda.memory_allocated() / 2**30:.1f} GB' if device.type == 'cuda' else ''
    print(f'    extracted {path.stem}: RAM {psutil.Process().memory_info().rss / 2**30:.1f} GB{gpu}')
    return X


C_GRID = (1e-3, 1e-2, 1e-1, 1.0, 10.0)


def choose_threshold(labels, probs):
    """Maximize recall subject to precision >= 0.90; ties go to the middle of the tied thresholds."""
    thresholds = np.round(np.arange(0.10, 0.901, 0.01), 2)
    scores = np.array([(precision_score(labels, probs >= t, zero_division=0) >= 0.90) * recall_score(labels, probs >= t, zero_division=0)
                       for t in thresholds])
    if scores.max() == 0:
        return 0.50
    tied = np.flatnonzero(scores == scores.max())
    return float(thresholds[tied[len(tied) // 2]])


def make_clf(C):
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))


def fit_head(X_tr, y_tr, g_tr, return_oof=False):
    """Pick C by 3-fold group CV over training clusters (ROC-AUC); threshold from those inner predictions."""
    inner = list(StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED).split(X_tr, y_tr, g_tr))
    best = (-1.0, None, None)
    for C in C_GRID:
        oof, aucs = np.full(len(y_tr), np.nan), []
        for tr, va in inner:
            if len(np.unique(y_tr[tr])) < 2:
                continue  # an inner training split with one class cannot be fit
            oof[va] = make_clf(C).fit(X_tr[tr], y_tr[tr]).predict_proba(X_tr[va])[:, 1]
            if len(np.unique(y_tr[va])) == 2:
                aucs.append(roc_auc_score(y_tr[va], oof[va]))
        score = np.mean(aucs) if aucs else 0.5
        if score > best[0]:
            best = (score, C, oof)
    _, C, oof = best
    ok = ~np.isnan(oof)
    threshold = choose_threshold(y_tr[ok], oof[ok]) if ok.any() and len(np.unique(y_tr[ok])) == 2 else 0.5
    clf = make_clf(C).fit(X_tr, y_tr)
    return (clf, threshold, oof) if return_oof else (clf, threshold)


def oof_probs(keys, idx, folds):
    """Out-of-fold probabilities and thresholds for originals `idx`, split by `folds` (same length as idx)."""
    X = np.hstack([extract(*k) for k in keys])[idx]
    y, g = orig.label.values[idx], orig.group_cluster.values[idx]
    probs, thr = np.zeros(len(idx)), np.zeros(len(idx))
    for fold in np.unique(folds):
        te = folds == fold
        clf, thr[te] = fit_head(X[~te], y[~te], g[~te])
        probs[te] = clf.predict_proba(X[te])[:, 1]
    return probs, thr


def as_rows(keys, idx, folds, probs, thr, scheme, model=None):
    return pd.DataFrame({'scheme': scheme, 'model': model or '+'.join(k[0] for k in keys),
                         'input': '+'.join(f'{k[1]}@{k[2]}' for k in keys), 'fold': folds,
                         'file': orig.file.values[idx], 'label': orig.label.values[idx], 'prob': probs,
                         'threshold': thr, 'group_cluster': orig.group_cluster.values[idx]})


def run_screen(keys, fold_col='fold_cluster'):
    """OOF rows for one feature set; keys = list of (backbone, variant, size), concatenated for ensembles."""
    idx, folds = np.arange(len(orig)), orig[fold_col].values
    return as_rows(keys, idx, folds, *oof_probs(keys, idx, folds), scheme={'fold_cluster': 'cluster', 'fold_s2': 'cluster_s2'}.get(fold_col, fold_col[5:]))

In [12]:
def _cluster_draws(df, B, seed):
    rng = np.random.default_rng(seed)
    groups = {c: [g.index.values for _, g in df[df.label == c].groupby('group_cluster')] for c in (0, 1)}
    for _ in range(B):
        yield np.concatenate([groups[c][j] for c in (0, 1) for j in rng.integers(0, len(groups[c]), len(groups[c]))])


def cluster_bootstrap_auc(df, B=2000, seed=42):
    df = df.reset_index(drop=True)
    aucs = [roc_auc_score(df.label.values[i], df.prob.values[i]) for i in _cluster_draws(df, B, seed)]
    return (float(roc_auc_score(df.label, df.prob)), *np.percentile(aucs, [2.5, 97.5]).tolist())


def paired_delta_auc(df_a, df_b, B=2000, seed=42):
    both = df_a.merge(df_b[['file', 'prob']], on='file', suffixes=('', '_b'), validate='one_to_one').reset_index(drop=True)
    d = lambda i: roc_auc_score(both.label.values[i], both.prob.values[i]) - roc_auc_score(both.label.values[i], both.prob_b.values[i])
    deltas = [d(i) for i in _cluster_draws(both, B, seed)]
    return (float(d(np.arange(len(both)))), *np.percentile(deltas, [2.5, 97.5]).tolist())


def summarize(run):
    auc, lo, hi = cluster_bootstrap_auc(run, B=300 if SMOKE else 2000)
    person = run.groupby('group_cluster').agg(label=('label', 'first'), prob=('prob', 'mean'))
    pred = run.prob >= run.threshold
    return {'model': run.model.iat[0], 'input': run.input.iat[0], 'scheme': run.scheme.iat[0],
            'auc_pooled': round(auc, 3), 'auc_lo': round(lo, 3), 'auc_hi': round(hi, 3),
            'auc_fold_mean': round(run.groupby('fold')[['label', 'prob']].apply(lambda g: roc_auc_score(g.label, g.prob)).mean(), 3),
            'auc_person': round(roc_auc_score(person.label, person.prob), 3),
            'sens': round(float(pred[run.label == 1].mean()), 3), 'spec': round(float((~pred)[run.label == 0].mean()), 3)}

In [13]:
# Self-checks: single-class inner split does not crash the head; DINOv2 output size at 224 and 448; TTA = mean of views
_X = np.random.default_rng(0).normal(size=(12, 4)); _y = np.array([0] * 6 + [1] * 6); _g = np.repeat(np.arange(4), 3)
_clf, _t = fit_head(_X, _y, _g)
assert 0.1 <= _t <= 0.9
_m, _norm = load_backbone('dinov2_s')
with torch.no_grad():
    for _s in (224, 448):
        assert _m(torch.zeros(1, 3, _s, _s, device=device)).shape == (1, 384), _s
_one = orig.iloc[:1]
_img = prepare(_one.file.iat[0], _one.family.iat[0], 'circle')
with torch.no_grad():
    _manual = _m(torch.stack([_norm(v) for v in views(_img, 224)]).to(device)).float().mean(0).cpu().numpy()
_saved, orig = orig, _one
try:
    _cached = OUT / 'features' / f'dinov2_s__circle__224{"_smoke" if SMOKE else ""}.npy'
    _cached.unlink(missing_ok=True)
    assert np.allclose(extract('dinov2_s', 'circle', 224)[0], _manual, atol=1e-4)
    orig = _saved
    try:  # the one-image cache must not be served to the full manifest
        extract('dinov2_s', 'circle', 224)
        raise RuntimeError('stale cache not detected')
    except AssertionError:
        pass
finally:
    orig = _saved
    _cached.unlink(missing_ok=True)
    _cached.with_suffix('.files.txt').unlink(missing_ok=True)
_bg = np.array(prepare(orig.file.iat[0], orig.family.iat[0], 'bg'))
_m0 = cv2.imread(str(OUT / 'masks' / f'{orig.family.iat[0]}.png'), 0) > 0
assert (_bg[_m0] == MEAN_RGB).all() and (_bg[~_m0] != MEAN_RGB).any(), 'surroundings input must blank the tongue'
del _m, _norm, _manual, _bg, _m0
_models.clear(); gc.collect()
print('self-checks ok')

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

    extracted dinov2_s__circle__224: RAM 1.5 GB, GPU 0.1 GB
self-checks ok


In [14]:
backbones = ['resnet50', 'dinov2_s'] if SMOKE else list(BACKBONES)
sizes = [224] if SMOKE else [224, 448]


def select(score, log=print):
    """Screening steps 1-5 with any score(keys) -> AUC; returns (finalists, winner name)."""
    log('Step 1: backbones on circle @224')
    step1 = {b: score([(b, 'circle', 224)]) for b in backbones}
    top2 = sorted(step1, key=step1.get, reverse=True)[:2]
    best = {b: (b, 'circle', 224) for b in top2}
    log('Step 2: input variants')
    for b in top2:
        for v in ('full', 'seg', 'segcrop'):
            if score([(b, v, 224)]) > score([best[b]]):
                best[b] = (b, v, 224)
    log('Step 3: gray-world white balance')
    for b in top2:
        gw = (b, best[b][1] + '+gw', best[b][2])
        if score([gw]) > score([best[b]]):
            best[b] = gw
    log('Step 4: resolution')
    for b in top2:
        for s in sizes[1:]:
            if score([(b, best[b][1], s)]) > score([best[b]]):
                best[b] = (b, best[b][1], s)
    log('Step 5: ensemble of the two finalists')
    finalists = {b: [best[b]] for b in top2}
    finalists['ensemble'] = [best[b] for b in top2]
    auc = {name: score(keys) for name, keys in finalists.items()}
    return finalists, max(auc, key=auc.get)


# (a) Leaderboard: every candidate scored on all originals with the identity-cluster folds
results, summary = {}, []


def score_full(keys, fold_col='fold_cluster'):
    key = (tuple(keys), fold_col)
    if key not in results:
        results[key] = run_screen(list(keys), fold_col)
        s = summarize(results[key]); summary.append(s)
        print(f"  {s['model']:<20} {s['input']:<28} {s['scheme']:<11} AUC {s['auc_pooled']:.3f} "
              f"[{s['auc_lo']:.3f}, {s['auc_hi']:.3f}]  person {s['auc_person']:.3f}")
    return float(roc_auc_score(results[key].label, results[key].prob))


finalists, winner = select(score_full)
print('Corners-only check of the finalists')
for name, keys in finalists.items():
    score_full([(b, 'corners' + ('+gw' if v.endswith('+gw') else ''), s) for b, v, s in keys])

# (b) Honest estimate: nested selection. For each fresh outer fold (fold_s2), steps 1-5 are re-run on the outer
# training people only, scored by inner group CV; the chosen pipeline then predicts the held-out people. The
# leaderboard above picked its winner on all 106 people, so re-scoring that winner on fold_s2 would reuse them.
y_all, g_all, s2 = orig.label.values, orig.group_cluster.values, orig.fold_s2.values
nested, picks = [], []
for f in range(N_FOLDS):
    tr, te = np.where(s2 != f)[0], np.where(s2 == f)[0]
    inner = np.zeros(len(tr), int)
    for k, (_, va) in enumerate(StratifiedGroupKFold(3, shuffle=True, random_state=SEED).split(tr, y_all[tr], g_all[tr])):
        inner[va] = k
    cache = {}
    def inner_score(keys):
        if tuple(keys) not in cache:
            cache[tuple(keys)] = roc_auc_score(y_all[tr], oof_probs(list(keys), tr, inner)[0])
        return cache[tuple(keys)]
    fin_f, win_f = select(inner_score, log=lambda *a: None)
    keys = fin_f[win_f]
    X = np.hstack([extract(*k) for k in keys])
    clf, t = fit_head(X[tr], y_all[tr], g_all[tr])
    picks.append('+'.join(f'{k[0]}:{k[1]}@{k[2]}' for k in keys))
    nested.append(as_rows(keys, te, np.full(len(te), f), clf.predict_proba(X[te])[:, 1], np.full(len(te), t),
                          scheme='cluster_s2', model='nested_selection'))
    print(f'  outer fold {f}: picked {picks[-1]}')
nested = pd.concat(nested, ignore_index=True)
nested['input'] = 'per-fold pick'
assert nested.file.is_unique and set(nested.file) == set(orig.file), 'nested OOF must cover every original once'
base_s2 = run_screen([('resnet50', 'circle', 224)], 'fold_s2')  # fixed pipeline, no selection
summary.append(summarize(nested)); summary.append(summarize(base_s2))
d, lo, hi = paired_delta_auc(nested, base_s2, B=300 if SMOKE else 2000)
print(f"Nested selection AUC {summary[-2]['auc_pooled']:.3f} [{summary[-2]['auc_lo']:.3f}, {summary[-2]['auc_hi']:.3f}] vs "
      f"frozen ResNet-50 circle {summary[-1]['auc_pooled']:.3f}: delta {d:+.3f} [{lo:+.3f}, {hi:+.3f}]")

pd.concat(list(results.values()) + [nested, base_s2], ignore_index=True).to_csv(OUT / 'screen_oof.csv', index=False)
summary_df = pd.DataFrame(summary)
summary_df.to_csv(OUT / 'screen_summary.csv', index=False)
json.dump({'leaderboard_winner': winner, 'finalists': {k: [list(x) for x in v] for k, v in finalists.items()},
           'nested_picks_per_outer_fold': picks,
           'nested_delta_vs_resnet50_circle': [round(d, 3), round(lo, 3), round(hi, 3)]},
          open(OUT / 'screen_choice.json', 'w'), indent=2)
print(summary_df.sort_values(['scheme', 'auc_pooled'], ascending=[True, False]).to_string(index=False))

Step 1: backbones on circle @224
    extracted dinov2_s__circle__224: RAM 1.6 GB, GPU 0.1 GB
  dinov2_s             circle@224                   cluster     AUC 0.835 [0.758, 0.907]  person 0.811
  dinov2_b             circle@224                   cluster     AUC 0.910 [0.853, 0.957]  person 0.906
  convnext_s           circle@224                   cluster     AUC 0.855 [0.778, 0.919]  person 0.828
  effv2_s              circle@224                   cluster     AUC 0.839 [0.760, 0.904]  person 0.816
  resnet50             circle@224                   cluster     AUC 0.883 [0.826, 0.933]  person 0.856
Step 2: input variants
  dinov2_b             full@224                     cluster     AUC 0.889 [0.831, 0.939]  person 0.892


model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

    extracted dinov2_b__seg__224__m099d67c9: RAM 2.0 GB, GPU 0.3 GB
  dinov2_b             seg@224                      cluster     AUC 0.896 [0.837, 0.946]  person 0.885
    extracted dinov2_b__segcrop__224__m099d67c9: RAM 2.0 GB, GPU 0.3 GB
  dinov2_b             segcrop@224                  cluster     AUC 0.909 [0.859, 0.952]  person 0.917
  resnet50             full@224                     cluster     AUC 0.890 [0.826, 0.945]  person 0.879
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 244MB/s]


    extracted resnet50__seg__224__m099d67c9: RAM 1.9 GB, GPU 0.1 GB
  resnet50             seg@224                      cluster     AUC 0.892 [0.835, 0.940]  person 0.886
    extracted resnet50__segcrop__224__m099d67c9: RAM 1.9 GB, GPU 0.1 GB
  resnet50             segcrop@224                  cluster     AUC 0.888 [0.831, 0.936]  person 0.890
Step 3: gray-world white balance
  dinov2_b             circle+gw@224                cluster     AUC 0.898 [0.847, 0.942]  person 0.893
    extracted resnet50__seg+gw__224__m099d67c9: RAM 1.9 GB, GPU 0.1 GB
  resnet50             seg+gw@224                   cluster     AUC 0.900 [0.850, 0.944]  person 0.884
Step 4: resolution
  dinov2_b             circle@448                   cluster     AUC 0.912 [0.856, 0.958]  person 0.911
    extracted resnet50__seg+gw__448__m099d67c9: RAM 2.0 GB, GPU 0.1 GB
  resnet50             seg+gw@448                   cluster     AUC 0.893 [0.833, 0.943]  person 0.866
Step 5: ensemble of the two finalists
  dinov2_b

... (2 outputs left out)

    extracted convnext_s__seg__224__m099d67c9: RAM 2.1 GB, GPU 0.2 GB
    extracted convnext_s__segcrop__224__m099d67c9: RAM 2.1 GB, GPU 0.2 GB
    extracted dinov2_b__seg+gw__224__m099d67c9: RAM 2.1 GB, GPU 0.3 GB
    extracted convnext_s__seg+gw__224__m099d67c9: RAM 2.1 GB, GPU 0.2 GB
    extracted dinov2_b__seg__448__m099d67c9: RAM 2.1 GB, GPU 0.4 GB
    extracted convnext_s__seg__448__m099d67c9: RAM 2.1 GB, GPU 0.2 GB
  outer fold 2: picked dinov2_b:seg@448


model.safetensors: reconstructing file:   0%|          |  0.00B / 86.5MB            

model.safetensors: downloading bytes:           |  0.00B            

    extracted effv2_s__seg__224__m099d67c9: RAM 2.0 GB, GPU 0.1 GB
    extracted effv2_s__segcrop__224__m099d67c9: RAM 2.0 GB, GPU 0.1 GB
  outer fold 3: picked dinov2_b:segcrop@224+effv2_s:full+gw@224
  outer fold 4: picked dinov2_b:full@224+convnext_s:circle@224
Nested selection AUC 0.860 [0.789, 0.924] vs frozen ResNet-50 circle 0.888: delta -0.028 [-0.084, +0.026]
            model                      input     scheme  auc_pooled  auc_lo  auc_hi  auc_fold_mean  auc_person  sens  spec
         dinov2_b                 circle@448    cluster       0.912   0.856   0.958          0.909       0.911 0.798 0.904
         dinov2_b                 circle@224    cluster       0.910   0.853   0.957          0.901       0.906 0.789 0.880
         dinov2_b                segcrop@224    cluster       0.909   0.859   0.952          0.891       0.917 0.765 0.868
         resnet50                 seg+gw@224    cluster       0.900   0.850   0.944          0.886       0.884 0.704 0.876
         dinov

## Final pipeline: tongue + surroundings

Frozen ResNet-50 on two inputs, the tongue mask (`seg`) and its surroundings (`bg`: everything outside the mask dilated by ~5% of the crop, i.e. lips, skin and teeth), features concatenated into one logistic head. The surroundings alone predict the label almost as well as the tongue, and photos were collected under one protocol, so the whole oral region is used and the tongue's own share is reported separately: the paired AUC difference of tongue + surroundings over surroundings alone.

This pipeline was chosen after the screening above, so it is reported on both fold sets (`fold_cluster`, `fold_s2`) against the fixed frozen ResNet-50 circle baseline and against each half alone. There is no independent external set: the Kaggle "external" tongue set is a re-export of the same collection. Sensitivity and specificity use the threshold chosen on small inner splits (about 8 people per class), so they are noisy; compare pipelines by AUC. Outputs: `final_oof.csv`, `final_summary.csv`, `final_pipeline.json`.

In [15]:
FINAL = [('resnet50', 'bg', 224), ('resnet50', 'seg', 224)]
parts = {'baseline circle': [('resnet50', 'circle', 224)], 'tongue (seg)': FINAL[1:],
         'surroundings (bg)': FINAL[:1], 'tongue + surroundings': FINAL}
final_rows, final_summary, final_deltas = [], [], {}
for fold_col in ('fold_cluster', 'fold_s2'):
    runs = {name: run_screen(keys, fold_col) for name, keys in parts.items()}
    for name, run in runs.items():
        final_rows.append(run.assign(pipeline=name))
        final_summary.append({'pipeline': name, **summarize(run)})
    for other in ('baseline circle', 'surroundings (bg)', 'tongue (seg)'):
        d, lo, hi = paired_delta_auc(runs['tongue + surroundings'], runs[other], B=300 if SMOKE else 2000)
        final_deltas[f'{fold_col}: tongue + surroundings - {other}'] = [round(d, 3), round(lo, 3), round(hi, 3)]
final_df = pd.DataFrame(final_summary)
print(final_df[['pipeline', 'scheme', 'auc_pooled', 'auc_lo', 'auc_hi', 'auc_person', 'sens', 'spec']].to_string(index=False))
for k, (d, lo, hi) in final_deltas.items():
    print(f'{k}: {d:+.3f} [{lo:+.3f}, {hi:+.3f}]')
pd.concat(final_rows, ignore_index=True).to_csv(OUT / 'final_oof.csv', index=False)
final_df.to_csv(OUT / 'final_summary.csv', index=False)
json.dump({'pipeline': [list(k) for k in FINAL], 'paired_delta_auc': final_deltas},
          open(OUT / 'final_pipeline.json', 'w'), indent=2)

    extracted resnet50__bg__224__m099d67c9: RAM 2.0 GB, GPU 0.1 GB
             pipeline     scheme  auc_pooled  auc_lo  auc_hi  auc_person  sens  spec
      baseline circle    cluster       0.883   0.826   0.933       0.856 0.449 0.944
         tongue (seg)    cluster       0.892   0.835   0.940       0.886 0.672 0.912
    surroundings (bg)    cluster       0.873   0.810   0.928       0.858 0.534 0.952
tongue + surroundings    cluster       0.922   0.874   0.961       0.906 0.721 0.916
      baseline circle cluster_s2       0.888   0.833   0.937       0.859 0.433 0.952
         tongue (seg) cluster_s2       0.860   0.797   0.916       0.862 0.717 0.880
    surroundings (bg) cluster_s2       0.848   0.774   0.907       0.838 0.494 0.948
tongue + surroundings cluster_s2       0.900   0.844   0.944       0.887 0.721 0.896
fold_cluster: tongue + surroundings - baseline circle: +0.039 [+0.006, +0.075]
fold_cluster: tongue + surroundings - surroundings (bg): +0.048 [+0.019, +0.081]
fold_clu

## Final model file

Fits the final head (scaler + logistic regression, `C` by group CV over clusters, threshold from the inner predictions) on all originals and saves it as `final_model.joblib` for `predict_tongue.py`. The backbone is torchvision's pretrained ResNet-50 (`IMAGENET1K_V2`, not trained here), so only the head is saved. Its expected performance is the cross-validated AUC above; fitting on everyone adds no new test. Checked on one photo collection only; not for diagnosis.

In [ ]:
import joblib, sklearn

X_all = np.hstack([extract(*k) for k in FINAL])
final_clf, final_thr = fit_head(X_all, orig.label.values, orig.group_cluster.values)
cv = final_df.set_index(['pipeline', 'scheme']).loc['tongue + surroundings']
bundle = {
    'clf': final_clf, 'threshold': final_thr, 'C': float(final_clf[-1].C),
    'threshold_rule': 'max recall with precision >= 0.90 on inner group-CV predictions; 0.50 when no threshold reaches 0.90',
    'features': [list(k) for k in FINAL],  # concatenation order: surroundings (bg) then tongue (seg)
    'backbone': 'torchvision resnet50 IMAGENET1K_V2, fc removed, ImageNet normalization',
    'views': 'FiveCrop at 90% x horizontal flip, mean embedding' if not SMOKE else 'smoke: centre x flip',
    'mask': {'work': WORK, 'min_area': MIN_AREA, 'max_area': MAX_AREA, 'rng_seed': 0, 'bg_dilate': 0.05},
    'mask_digest': MASK_TAG, 'n_train': len(orig), 'n_clusters': int(orig.group_cluster.nunique()),
    'cv_auc': {s: [float(cv.loc[s, c]) for c in ('auc_pooled', 'auc_lo', 'auc_hi')] for s in cv.index},
    'versions': {'torch': torch.__version__, 'sklearn': sklearn.__version__, 'opencv': cv2.__version__},
}
joblib.dump(bundle, OUT / 'final_model.joblib')
reloaded = joblib.load(OUT / 'final_model.joblib')
assert np.allclose(reloaded['clf'].predict_proba(X_all[:5]), final_clf.predict_proba(X_all[:5]))
print(f"final_model.joblib: C={bundle['C']}, threshold={final_thr:.2f}, trained on {len(orig)} photos / "
      f"{bundle['n_clusters']} people, CV AUC {bundle['cv_auc']}")

In [ ]:
# [paper-analyses-T1]
# Paper analyses (2026-11-14 reviews): cheap checks on the cached frozen features; results in paper_analyses.json
AN = {}
REG = {'circle': [('resnet50', 'circle', 224)], 'seg': [('resnet50', 'seg', 224)],
       'bg': [('resnet50', 'bg', 224)], 'seg+bg': FINAL}
B_BOOT = 300 if SMOKE else 2000
_rs = {}


def rs(keys, fold_col):
    """Memoized run_screen."""
    k = (tuple(map(tuple, keys)), fold_col)
    if k not in _rs:
        _rs[k] = run_screen(list(keys), fold_col)
    return _rs[k]


def ci(run):
    s = summarize(run)
    return {'auc': [s['auc_pooled'], s['auc_lo'], s['auc_hi']], 'auc_fold_mean': s['auc_fold_mean'], 'auc_person': s['auc_person']}


def delta(a, b):
    return [round(x, 3) for x in paired_delta_auc(a, b, B=B_BOOT)]


def save_an():
    json.dump(AN, open(OUT / 'paper_analyses.json', 'w'), indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o))


def nested_region(candidates, fold_col='fold_s2'):
    """Per outer fold, pick the input region with the best inner 3-fold group-CV AUC on the outer-training people."""
    y, g, folds = orig.label.values, orig.group_cluster.values, orig[fold_col].values
    rows, picks = [], []
    for f in np.unique(folds):
        tr, te = np.where(folds != f)[0], np.where(folds == f)[0]
        inner = np.zeros(len(tr), int)
        for k, (_, va) in enumerate(StratifiedGroupKFold(3, shuffle=True, random_state=SEED).split(tr, y[tr], g[tr])):
            inner[va] = k
        score = {n: roc_auc_score(y[tr], oof_probs(keys, tr, inner)[0]) for n, keys in candidates.items()}
        pick = max(score, key=score.get)
        picks.append(pick)
        X = np.hstack([extract(*k) for k in candidates[pick]])
        clf, t = fit_head(X[tr], y[tr], g[tr])
        rows.append(as_rows(candidates[pick], te, np.full(len(te), f), clf.predict_proba(X[te])[:, 1],
                            np.full(len(te), t), scheme={'fold_s2': 'cluster_s2'}.get(fold_col, fold_col[5:]), model='nested_region'))
    out = pd.concat(rows, ignore_index=True)
    out['input'] = 'per-fold pick'
    return out, picks


# Self-checks (Task 1): HSV baseline features and nested region selection
_H = extract('hsv', 'circle', 224)
assert _H.shape == (len(orig), 960) and np.allclose(_H.sum(1), 1, atol=1e-6), 'hsv features: 960 bins, L1-normalized'
_nr, _picks = nested_region(REG, 'fold_s2')
assert _nr.file.is_unique and set(_nr.file) == set(orig.file), 'nested region OOF must cover every original once'

AN['nested_region'] = {}
for fold_col in ('fold_cluster', 'fold_s2'):
    nr, picks = (_nr, _picks) if fold_col == 'fold_s2' else nested_region(REG, fold_col)
    AN['nested_region'][fold_col] = {**ci(nr), 'picks': picks, 'delta_vs_circle': delta(nr, rs(REG['circle'], fold_col))}
_ss = OUT / 'screen_summary.csv'
if _ss.exists():  # written by the screening cell; absent when that cell is skipped (local checks)
    sdf = pd.read_csv(_ss)
    lb = sdf[sdf.scheme == 'cluster']
    get = lambda m, i: float(lb[(lb.model == m) & (lb.input == i)].auc_pooled.iloc[0])
    AN['screening'] = {'n_configs': int(len(lb)), 'leaderboard_max': float(lb.auc_pooled.max()),
                       'leaderboard_argmax': lb.loc[lb.auc_pooled.idxmax(), ['model', 'input']].tolist(),
                       'nested': float(sdf[sdf.model == 'nested_selection'].auc_pooled.iloc[0]),
                       'gray_world_delta': {'dinov2_b circle@224': round(get('dinov2_b', 'circle+gw@224') - get('dinov2_b', 'circle@224'), 3),
                                            'resnet50 segcrop@224': round(get('resnet50', 'segcrop+gw@224') - get('resnet50', 'segcrop@224'), 3)}}
save_an()
print(json.dumps({k: AN[k] for k in AN}, indent=1, default=str)[:1500])

In [ ]:
# [paper-analyses-T2]
# Tongue-specific signal controls (M5, M9): capacity-matched permuted tongue features, paired region deltas,
# a low-level HSV baseline on the surroundings, and the within-class correlation of tongue-only vs surroundings-only scores
def permute_within(X, parts, seed=0):
    """Copy of X whose rows are shuffled inside each index array in `parts` (fixed seed)."""
    rng, Xp = np.random.default_rng(seed), X.copy()
    for part in parts:
        Xp[part] = X[rng.permutation(part)]
    return Xp


def permuted_control(fold_col, seed=0):
    """OOF rows for [surroundings, tongue features shuffled across images]: same dimension and head, no tongue-label link."""
    Xb, Xs = extract(*REG['bg'][0]), extract(*REG['seg'][0])
    y, g, folds = orig.label.values, orig.group_cluster.values, orig[fold_col].values
    probs, thr = np.zeros(len(y)), np.zeros(len(y))
    for f in np.unique(folds):
        tr, te = folds != f, folds == f
        X = np.hstack([Xb, permute_within(Xs, [np.where(tr)[0], np.where(te)[0]], seed + int(f))])
        clf, thr[te] = fit_head(X[tr], y[tr], g[tr])
        probs[te] = clf.predict_proba(X[te])[:, 1]
    return as_rows(REG['bg'] + REG['seg'], np.arange(len(y)), folds, probs, thr,
                   scheme={'fold_s2': 'cluster_s2'}.get(fold_col, fold_col[5:]), model='bg+seg_perm')


# Self-checks (Task 2): row permutation keeps every column's values and breaks the row order
_X = np.arange(40, dtype=float).reshape(10, 4)
_Xp = permute_within(_X, [np.arange(6), np.arange(6, 10)], seed=0)
assert np.allclose(np.sort(_Xp, 0), np.sort(_X, 0)) and not np.allclose(_Xp, _X), 'permutation must reorder rows only'
assert np.allclose(np.sort(_Xp[:6], 0), np.sort(_X[:6], 0)), 'rows must stay inside their part (train vs test)'

AN['signal'] = {}
for fold_col in ('fold_cluster', 'fold_s2'):
    r = {n: rs(k, fold_col) for n, k in REG.items()}
    r['hsv:bg'] = rs([('hsv', 'bg', 224)], fold_col)
    perm = permuted_control(fold_col)
    sp = lambda lab: float(pd.Series(r['seg'].prob[r['seg'].label == lab].values).corr(
        pd.Series(r['bg'].prob[r['bg'].label == lab].values), method='spearman'))
    AN['signal'][fold_col] = {
        'auc': {n: ci(run)['auc'] for n, run in {**r, 'bg+seg_perm': perm}.items()},
        'seg+bg - bg+seg_perm': delta(r['seg+bg'], perm),
        'seg+bg - bg': delta(r['seg+bg'], r['bg']),
        'seg - bg': delta(r['seg'], r['bg']),
        'bg - hsv:bg': delta(r['bg'], r['hsv:bg']),
        'spearman_seg_vs_bg_within_class': {'non_T2D': round(sp(0), 3), 'T2D': round(sp(1), 3)},
    }
save_an()
print(json.dumps(AN['signal'], indent=1)[:2500])

In [ ]:
# [paper-analyses-T3]
# Grouping sensitivity (M3): coarser presumed participants, by linking each identity cluster to its most similar
# same-class cluster in the surroundings embedding (lips, skin, clothing: a person cue independent of tongue colour)
def coarsen_clusters():
    """Halve the identity clusters within each class: average-linkage agglomeration of the clusters' mean
    surroundings embeddings (cosine), so every coarse group joins about two similar same-class clusters."""
    from sklearn.cluster import AgglomerativeClustering
    E = extract(*REG['bg'][0])
    E = E - E.mean(0)  # pooled ReLU features are all positive; centre so cosine spreads
    cl, y = orig.group_cluster.values, orig.label.values
    out = np.empty(len(cl), dtype=object)
    for lab in (0, 1):
        ids = np.unique(cl[y == lab])
        M = np.stack([E[cl == c].mean(0) for c in ids])
        k = int(np.ceil(len(ids) / 2))
        merged = AgglomerativeClustering(n_clusters=k, metric='cosine', linkage='average').fit_predict(M)
        for c, m in zip(ids, merged):
            out[cl == c] = f'{lab}_{m}'
    return out


# Self-checks (Task 3): coarse groups are single-label, contain whole clusters, and are fewer than clusters
_coarse = coarsen_clusters()
assert orig.groupby(_coarse).label.nunique().max() == 1, 'coarse groups must not mix classes'
assert pd.Series(_coarse).groupby(orig.group_cluster.values).nunique().max() == 1, 'a cluster must map to one coarse group'
assert len(np.unique(_coarse)) < orig.group_cluster.nunique(), 'coarsening must reduce the group count'

orig['group_coarse'] = _coarse
orig['fold_coarse'] = -1
for k, (_, te) in enumerate(StratifiedGroupKFold(N_FOLDS, shuffle=True, random_state=SEED).split(orig, orig.label, orig.group_coarse)):
    orig.loc[orig.index[te], 'fold_coarse'] = k
assert orig.groupby('fold_coarse').label.nunique().min() == 2, 'every coarse fold needs both classes'
sizes = orig.groupby(['label', 'group_cluster']).size()
ladder = {f: rs(REG['circle'], f) for f in ('fold_photo', 'fold_block', 'fold_cluster', 'fold_coarse')}
AN['grouping'] = {
    'cluster_size_hist': {('T2D' if lab else 'non_T2D'): sizes[lab].value_counts().sort_index().to_dict() for lab in (0, 1)},
    'n_clusters': {('T2D' if lab else 'non_T2D'): int(sizes[lab].shape[0]) for lab in (0, 1)},
    'n_coarse': {('T2D' if lab else 'non_T2D'): int(orig[orig.label == lab].group_coarse.nunique()) for lab in (0, 1)},
    'frozen_resnet50_circle': {f[5:]: ci(r) for f, r in ladder.items()},
    'photo - cluster': delta(ladder['fold_photo'], ladder['fold_cluster']),
    'block - cluster': delta(ladder['fold_block'], ladder['fold_cluster']),
    'cluster - coarse': delta(ladder['fold_cluster'], ladder['fold_coarse']),
    'final seg+bg on coarse': ci(rs(FINAL, 'fold_coarse')),
}
save_an()
print(json.dumps(AN['grouping'], indent=1, default=str)[:2500])

In [ ]:
# [paper-analyses-T4]
# Independent residual-leakage audit (M4): cosine similarity of surroundings embeddings, a cue the identity clustering
# (tongue colour histograms, SIFT on the crop) did not use. Cross-fold best match vs best match inside the same cluster.
def bg_leak_audit(fold_col):
    E = extract(*REG['bg'][0])
    E = E - E.mean(0)  # pooled ReLU features are all positive; centre so cosine spreads
    E = E / np.linalg.norm(E, axis=1, keepdims=True)
    S = E @ E.T
    np.fill_diagonal(S, -np.inf)
    folds, cl = orig[fold_col].values, orig.group_cluster.values
    cross_S = np.where(folds[:, None] != folds[None, :], S, -np.inf)
    within_S = np.where((cl[:, None] == cl[None, :]), S, -np.inf)
    within = within_S.max(1)
    per = pd.DataFrame({'file': orig.file, 'cross': cross_S.max(1), 'partner': cross_S.argmax(1),
                        'within': np.where(np.isfinite(within), within, np.nan)})
    med = float(np.nanmedian(per.within))
    return {'per_image': per, 'within_median': med, 'within_p05': float(np.nanpercentile(per.within, 5)),
            'cross_median': float(per.cross.median()), 'cross_p95': float(per.cross.quantile(.95)),
            'n_cross_above_within_median': int((per.cross > med).sum())}


# Self-checks (Task 4): every original gets a cross-fold similarity; singletons get NaN within-cluster
_aud = bg_leak_audit('fold_cluster')
assert len(_aud['per_image']) == len(orig) and _aud['per_image'].cross.notna().all()
_single = orig.group_cluster.map(orig.group_cluster.value_counts()) == 1
assert _aud['per_image'].within[_single.values].isna().all()

AN['audit_bg'] = {k: v for k, v in _aud.items() if k != 'per_image'}
# contact sheet of the 20 most similar cross-fold pairs (one row per pair: image | partner) for an eye check
top = _aud['per_image'].sort_values('cross', ascending=False)
top = top[~top[['partner']].assign(i=top.index).apply(lambda r: tuple(sorted((r.i, r.partner))), axis=1).duplicated()].head(20)
tiles = []
for i, r in top.iterrows():
    a, b = (cv2.resize(cv2.imread(str(root / orig.file.iat[j])), (160, 160)) for j in (i, int(r.partner)))
    t = np.hstack([a, b])
    cv2.putText(t, f'{r.cross:.3f}', (4, 16), 0, 0.5, (0, 255, 255), 1)
    tiles.append(t)
while len(tiles) % 4:
    tiles.append(np.zeros_like(tiles[0]))
cv2.imwrite(str(OUT / 'bg_audit_sheet.jpg'), np.vstack([np.hstack(tiles[k:k + 4]) for k in range(0, len(tiles), 4)]))
AN['audit_bg']['top_pairs'] = [[orig.file.iat[i], orig.file.iat[int(r.partner)], round(float(r.cross), 3)] for i, r in top.iterrows()]
save_an()
print({k: v for k, v in AN['audit_bg'].items() if k != 'top_pairs'})

In [ ]:
# [paper-analyses-T5]
# Split variance and operating point (M6, M8): repeated CV over fold seeds; sensitivity at 90% specificity with the
# threshold set inside each fold; how often the precision rule fell back to 0.50; calibration intercept and slope
from sklearn.linear_model import LogisticRegression


def repeated_cv(keys, seeds=range(10)):
    y, g, idx = orig.label.values, orig.group_cluster.values, np.arange(len(orig))
    out = []
    for s in seeds:
        folds = np.zeros(len(y), int)
        for k, (_, te) in enumerate(StratifiedGroupKFold(N_FOLDS, shuffle=True, random_state=s).split(idx, y, g)):
            folds[te] = k
        p = oof_probs(keys, idx, folds)[0]
        per = [roc_auc_score(y[folds == k], p[folds == k]) for k in range(N_FOLDS)]
        out.append({'seed': s, 'auc_pooled': roc_auc_score(y, p), 'auc_fold_mean': np.mean(per), 'auc_fold_sd': np.std(per, ddof=1)})
    return pd.DataFrame(out)


def calibration(y, p):
    """(intercept, slope) of an unpenalized logistic recalibration of y on logit(p)."""
    p = np.clip(p, 1e-6, 1 - 1e-6)
    lr = LogisticRegression(penalty=None, max_iter=1000).fit(np.log(p / (1 - p))[:, None], y)
    return float(lr.intercept_[0]), float(lr.coef_[0, 0])


def operating_point(keys, fold_col):
    X = np.hstack([extract(*k) for k in keys])
    y, g, folds = orig.label.values, orig.group_cluster.values, orig[fold_col].values
    pred, prob, fallback = np.zeros(len(y), bool), np.zeros(len(y)), 0
    grid = np.round(np.arange(0.10, 0.901, 0.01), 2)
    for f in np.unique(folds):
        tr, te = folds != f, folds == f
        clf, thr, oof = fit_head(X[tr], y[tr], g[tr], return_oof=True)
        ok = ~np.isnan(oof)
        neg = oof[ok & (y[tr] == 0)]
        prob[te] = clf.predict_proba(X[te])[:, 1]
        pred[te] = prob[te] >= np.quantile(neg, 0.90)
        fallback += max(precision_score(y[tr][ok], oof[ok] >= t, zero_division=0) for t in grid) < 0.90
    a, b = calibration(y, prob)
    return {'sens_at_spec90': round(float(pred[y == 1].mean()), 3), 'spec_achieved': round(float((~pred)[y == 0].mean()), 3),
            'precision_rule_fallback_folds': int(fallback), 'calib_intercept': round(a, 3), 'calib_slope': round(b, 3),
            'brier': round(float(np.mean((prob - y) ** 2)), 3)}


# Self-checks (Task 5): repeated CV shape; calibration recovers a calibrated synthetic model
_rc = repeated_cv(REG['circle'], seeds=range(3) if SMOKE else range(10))
assert len(_rc) == (3 if SMOKE else 10) and (_rc.auc_fold_sd > 0).all()
_rng = np.random.default_rng(0)
_p = _rng.uniform(0.02, 0.98, 5000)
_y = (_rng.uniform(size=5000) < _p).astype(int)
_a, _b = calibration(_y, _p)
assert abs(_b - 1) < 0.15 and abs(_a) < 0.15, (_a, _b)

seeds = range(3) if SMOKE else range(10)
AN['variance'] = {}
for n, keys in REG.items():
    r = _rc if n == 'circle' else repeated_cv(keys, seeds)
    AN['variance'][n] = {'pooled_mean': round(r.auc_pooled.mean(), 3), 'pooled_min': round(r.auc_pooled.min(), 3),
                         'pooled_max': round(r.auc_pooled.max(), 3), 'fold_sd_mean': round(r.auc_fold_sd.mean(), 3)}
AN['operating_point'] = {f: {n: operating_point(REG[n], f) for n in ('circle', 'seg+bg')} for f in ('fold_cluster', 'fold_s2')}
save_an()
print(json.dumps({k: AN[k] for k in ('variance', 'operating_point')}, indent=1))

In [ ]:
# [paper-analyses-T7]
# Region illustration for the paper (Fig. 2): one T2D and one non-T2D original as full / circle / corners / tongue / surroundings
_names = ['full', 'circle', 'corners', 'seg', 'bg']
_rows = []
for lab in (1, 0):
    r = orig[orig.label == lab].iloc[0]
    _rows.append(np.hstack([np.array(prepare(r.file, r.family, v).resize((200, 200))) for v in _names]))
cv2.imwrite(str(OUT / 'regions.png'), cv2.cvtColor(np.vstack(_rows), cv2.COLOR_RGB2BGR))
print('regions.png: rows T2D, non-T2D; columns', _names)